In [1]:
import torch
import math
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, IterableDataset


In [7]:
class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self,x):
        return x + self.pe[:, :x.size(1), :]

class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.layer_norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        residual = x
        x = self.layer_norm1(x)
        x, _ = self.self_attn(x, x, x)
        x = residual + x

        residual = x
        x = self.layer_norm2(x)
        x = self.ffn(x)
        x = residual + x
        return x

class AudioEncoder(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=2048, num_layers=8, dropout=0.1, n_mels=80):
        super().__init__()
        self.conv1 = nn.Conv1d(n_mels, d_model, kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv1d(d_model, d_model, kernel_size=3, stride=2, padding=1)
        self.gelu = nn.GELU()

        self.pos_encoding = SinusoidalPE(d_model)
        self.dropout = nn.Dropout(dropout)
        
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        self.ln_final = nn.LayerNorm(d_model)

    def forward(self, x):
        # input shape: (B, n_mels, T)
        x = self.gelu(self.conv1(x))
        x = self.gelu(self.conv2(x))

        x = x.permute(0,2,1)

        x = self.pos_encoding(x)
        x = self.dropout(x)
        for layer in self.layers:
            x = layer(x)
        x = self.ln_final(x)
        return x

In [8]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.layer_norm2 = nn.LayerNorm(d_model)
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.layer_norm3 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x, enc_out, causal_mask=None):
        residual = x
        x = self.layer_norm1(x)
        x, _ = self.self_attn(x, x, x, attn_mask=causal_mask)
        x = residual + x

        residual = x
        x = self.layer_norm2(x)
        x, _ = self.cross_attn(x, enc_out, enc_out)
        x = residual + x

        residual = x
        x = self.layer_norm3(x)
        x = self.ffn(x)
        x = residual + x
        return x

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, d_ff=2048, num_layers=8, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = SinusoidalPE(d_model)
        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            DecoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        self.ln_final = nn.LayerNorm(d_model)
        self.output_proj = nn.Linear(d_model, vocab_size)

    def forward(self, x, enc_out):
        x = self.token_emb(x)
        x = self.pos_encoding(x)
        x = self.dropout(x)

        seq_len = x.size(1)
        causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().to(x.device)

        for layer in self.layers:
            x = layer(x, enc_out, causal_mask)
        
        x = self.ln_final(x)
        logits = self.output_proj(x)
        return logits

In [9]:
class MiniWhisper(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, mel, tokens):
        enc_out = self.encoder(mel)
        logits = self.decoder(tokens, enc_out)
        return logits

In [2]:
seamless = load_dataset(
    "ai4bharat/SeamlessAlign", 
    name="indic2en", 
    split="hindi", 
    streaming=True
).shuffle(buffer_size=1000)


Resolving data files:   0%|          | 0/332 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/161 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/111 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/411 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/332 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/161 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/111 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/411 [00:00<?, ?it/s]

In [ ]:
from torchaudio.transforms import MelSpectrogram, AmplitudeToDB
from transformers import WhisperTokenizer

class WhisperStreamDataset(IterableDataset):
    def __init__(self, stream, tokenizer, sample_rate=16000, n_mels=80, max_audio_len=480000, max_tokens=256):
        self.stream = stream
        self.tokenizer = tokenizer
        self.max_audio_len = max_audio_len
        self.max_tokens = max_tokens

        self.mel_transform = MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=400,
            hop_length=160,
            n_mels=n_mels
        )
        self.amp_to_db = AmplitudeToDB()

        self.SOT = tokenizer.convert_tokens_to_ids("<|startoftranscript|>")
        self.EOT = tokenizer.convert_tokens_to_ids("<|endoftext|>")
        self.HI = tokenizer.convert_tokens_to_ids("<|hi|>")
        self.TRANSLATE = tokenizer.convert_tokens_to_ids("<|translate|>")

    def __iter__(self):
        for sample in self.stream:
            result = self.process(sample)
            if result is not None:
                yield result

    def process(self, sample):
        audio = sample["audio"]["array"]

        # Skip too long (>30s) or too short (<0.1s)
        if len(audio) > self.max_audio_len or len(audio) < 1600:
            return None

        waveform = torch.tensor(audio, dtype=torch.float32)
        mel = self.mel_transform(waveform)
        mel = self.amp_to_db(mel)  # (80, T)

        # Tokenize English translation
        text = sample["text"]
        token_ids = self.tokenizer.encode(text, add_special_tokens=False)

        if len(token_ids) > self.max_tokens - 4:
            return None

        # decoder_input: [SOT, HI, TRANSLATE, ...tokens]
        # labels:        [HI, TRANSLATE, ...tokens, EOT]
        decoder_input = [self.SOT, self.HI, self.TRANSLATE] + token_ids
        labels = [self.HI, self.TRANSLATE] + token_ids + [self.EOT]

        return mel, decoder_input, labels


def collate_fn(batch):
    mels, input_ids, labels = zip(*batch)

    # Pad mels to max time in batch
    max_t = max(m.size(1) for m in mels)
    mel_padded = torch.stack([F.pad(m, (0, max_t - m.size(1))) for m in mels])

    # Pad tokens (-100 for labels = ignored in cross_entropy)
    max_s = max(len(ids) for ids in input_ids)
    input_padded = torch.zeros(len(batch), max_s, dtype=torch.long)
    label_padded = torch.full((len(batch), max_s), -100, dtype=torch.long)

    for i, (inp, lab) in enumerate(zip(input_ids, labels)):
        input_padded[i, :len(inp)] = torch.tensor(inp)
        label_padded[i, :len(lab)] = torch.tensor(lab)

    return mel_padded, input_padded, label_padded


In [5]:
from transformers import WhisperTokenizer
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-large-v3")
tokenizer.vocab_size

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

50257

In [6]:
!pip install jiwer --quiet

In [7]:
from jiwer import wer, cer

truth = "hello world"
pred = "hello word"
print("WER:", wer(truth, pred))
print("CER:", cer(truth, pred))

WER: 0.5
CER: 0.09090909090909091
